In [22]:
from  ..middleWare import *
#初始化模型
load_dotenv(override=True)

DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')

custom_profile={'max_input_tokens':128000}
model=init_chat_model(
    model='deepseek-v4-flash',
    model_provider='deepseek',
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={'thinking':{"type":'disabled'}},
    profile=custom_profile
)
#中间件有很多，包括模型商提供的和langchain内置的
#创建智能体使用总结摘要中间件
#SummarizationMiddleware用于总结历史消息上下文，主要包括trigger,keep,summary_prompt等参数
#trigger包括三个参数，tokens，messages,fraction三个满足任一条件即触发总结
#其中fraction上下文比例需传入max_input_tokens参数
#keep为总结后保留的消息条数
#summary_prompt为触发总结时的自定义提示词，需要插入{messages}占位符
myagent=create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[
                ('tokens',100),
                ('messages',3),
                ('fraction',0.001)
            ],
            keep=('messages',2),
            # summary_prompt='对历史消息的摘要如下\n{messages}'
        )
    ]
)

In [23]:

messages=[
    SystemMessage('你是一个友好的AI助手,请用中文回答用户问题'),
    HumanMessage('你好，我是hyz，你是谁？'),
    AIMessage('你好hyz，我是AI助手'),
    HumanMessage('今天中午我吃了番茄炒蛋，你认为健康吗？'),
    AIMessage('番茄炒蛋是很美味健康的食物。'),
    HumanMessage('晚饭有推荐的食物吗？')
]
response=myagent.invoke({
    'messages':messages,
})
for msg in response['messages']:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
用户hyz与AI助手打招呼并自我介绍，询问AI身份，随后询问午餐（番茄炒蛋）是否健康。

## SUMMARY
- 用户名为hyz。
- AI助手身份已确认。
- 用户午餐内容为番茄炒蛋，正在咨询其健康性。未给出AI对该问题的答复。

## ARTIFACTS
None

## NEXT STEPS
- 回答用户关于番茄炒蛋是否健康的问题，提供营养分析和建议。
================================== Ai Message ==================================

番茄炒蛋是很美味健康的食物。
================================ Human Message =================================

晚饭有推荐的食物吗？
================================== Ai Message ==================================

既然你午餐吃了番茄炒蛋（营养均衡、富含维生素和蛋白质），晚餐建议以**清淡、易消化、补充膳食纤维**为原则，帮助身体夜间修复，同时避免积食。

以下是几个推荐方向，供你参考：

**1. 经典优化版：蛋白质+蔬菜组合**
-   **清蒸鱼（如鲈鱼、鳕鱼）** + **白灼西兰花/秋葵** + **少量杂粮饭（小米/糙米）**
-   **理由**：鱼肉（优质蛋白+不饱和脂肪酸）比红肉更易消化，清蒸做法油脂少；西兰花富含维C和纤维，很适合晚间食用。

**2. 快手清爽版：汤羹+粗粮**
-   **虾仁豆腐菌菇汤** + **蒸半个玉米/红薯**
-   **理由**：汤羹水分多、饱腹感强；虾仁+豆腐提供双重蛋白质，热量低；粗粮抗饿且升糖慢，更适合晚餐。

**3. 素食清肠版：蔬菜+豆制品**
-   **凉拌菠菜/黄瓜（少油少醋）** + **麻婆豆腐（少辣版）** 或 **清炒